# LAB | Feature Engineering

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In [29]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.preprocessing import MinMaxScaler, StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score

RANDOM_STATE = 42

In [30]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


**Check the shape of your data**

In [31]:
spaceship.shape

(8693, 14)

**Check for data types**

In [32]:
spaceship.dtypes

PassengerId         str
HomePlanet          str
CryoSleep        object
Cabin               str
Destination         str
Age             float64
VIP              object
RoomService     float64
FoodCourt       float64
ShoppingMall    float64
Spa             float64
VRDeck          float64
Name                str
Transported        bool
dtype: object

**Check for missing values**

In [33]:
spaceship.isna().sum()

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

There are multiple strategies to handle missing data

- Removing all rows or all columns containing missing data.
- Filling all missing values with a value (mean in continouos or mode in categorical for example).
- Filling all missing values with an algorithm.

For this exercise, because we have such low amount of null values, we will drop rows containing any missing value. 

In [34]:
spaceship = spaceship.dropna()
spaceship.shape

(6606, 14)

- **Cabin** is too granular - transform it in order to obtain {'A', 'B', 'C', 'D', 'E', 'F', 'G', 'T'}

In [35]:
# Spliting and selecting the first value of the column Cabin

spaceship["Cabin"] = spaceship["Cabin"].str.split("/").str[0]
print(spaceship["Cabin"].value_counts())

Cabin
F    2152
G    1973
E     683
B     628
C     587
D     374
A     207
T       2
Name: count, dtype: int64


- Drop PassengerId and Name

In [36]:
spaceship = spaceship.drop(columns = ["PassengerId","Name"])
spaceship

,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported
0,Europa,False,B,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,False
1,Earth,False,F,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,True
2,Europa,False,A,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,False
3,Europa,False,A,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,False
4,Earth,False,F,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,True
...,...,...,...,...,...,...,...,...,...,...,...,...
8688,Europa,False,A,55 Cancri e,41.0,True,0.0,6819.0,0.0,1643.0,74.0,False
8689,Earth,True,G,PSO J318.5-22,18.0,False,0.0,0.0,0.0,0.0,0.0,False
8690,Earth,False,G,TRAPPIST-1e,26.0,False,0.0,0.0,1872.0,1.0,0.0,True
8691,Europa,False,E,55 Cancri e,32.0,False,0.0,1049.0,0.0,353.0,3235.0,False


**Perform Train Test Split**

In [42]:
X = spaceship.drop(columns='Transported')
y = spaceship['Transported']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE,  stratify=y
)

print('Train:', X_train.shape, 'Test:', X_test.shape)

Train: (5284, 11) Test: (1322, 11)


- For non-numerical columns, do dummies.

In [48]:
# To analyse the categorical columns
cat_columns = spaceship.select_dtypes(exclude = "number")
print(cat_columns.head())
print(cat_columns["HomePlanet"].value_counts()) # Nominal categorical values
print(cat_columns["CryoSleep"].value_counts()) # could be Boolean
print(cat_columns["Cabin"].value_counts()) # Gives us the Deck number - Nominal categorical values 
print(cat_columns["Destination"].value_counts()) # Nominal categorical values
print(cat_columns["VIP"].value_counts()) # could be Boolean
# Transported is a target

  HomePlanet CryoSleep Cabin  Destination    VIP  Transported
0     Europa     False     B  TRAPPIST-1e  False        False
1      Earth     False     F  TRAPPIST-1e  False         True
2     Europa     False     A  TRAPPIST-1e   True        False
3     Europa     False     A  TRAPPIST-1e  False        False
4      Earth     False     F  TRAPPIST-1e  False         True
HomePlanet
Earth     3566
Europa    1673
Mars      1367
Name: count, dtype: int64
CryoSleep
False    4274
True     2332
Name: count, dtype: int64
Cabin
F    2152
G    1973
E     683
B     628
C     587
D     374
A     207
T       2
Name: count, dtype: int64
Destination
TRAPPIST-1e      4576
55 Cancri e      1407
PSO J318.5-22     623
Name: count, dtype: int64
VIP
False    6444
True      162
Name: count, dtype: int64


In [51]:
#Encoding the train data

encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
encoder.fit(X_train[['HomePlanet']])
encoded_train = pd.DataFrame(
    encoder.transform(X_train[['HomePlanet']]),
    columns=encoder.get_feature_names_out(['HomePlanet']),
    index=X_train.index,
)
display(encoded_train.head())

,HomePlanet_Earth,HomePlanet_Europa,HomePlanet_Mars
6017,0.0,1.0,0.0
8533,0.0,0.0,1.0
6961,0.0,1.0,0.0
2248,1.0,0.0,0.0
2422,0.0,0.0,1.0


In [52]:
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
encoder.fit(X_train[['Cabin']])
encoded_train = pd.DataFrame(
    encoder.transform(X_train[['Cabin']]),
    columns=encoder.get_feature_names_out(['Cabin']),
    index=X_train.index,
)
display(encoded_train.head())

,Cabin_A,Cabin_B,Cabin_C,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_T
6017,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8533,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
6961,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
2248,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2422,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


In [53]:
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
encoder.fit(X_train[['Destination']])
encoded_train = pd.DataFrame(
    encoder.transform(X_train[['Destination']]),
    columns=encoder.get_feature_names_out(['Destination']),
    index=X_train.index,
)
display(encoded_train.head())

,Destination_55 Cancri e,Destination_PSO J318.5-22,Destination_TRAPPIST-1e
6017,1.0,0.0,0.0
8533,0.0,0.0,1.0
6961,0.0,0.0,1.0
2248,1.0,0.0,0.0
2422,0.0,0.0,1.0


In [60]:
# Converting to Boolean
spaceship[["CryoSleep","VIP"]] =spaceship[["CryoSleep","VIP"]].astype(int)
cat_columns = spaceship.select_dtypes(exclude = "number") # should give no output


In [66]:
#Creating pipeline
# Branch 2: Nominal variables
nominal_pipeline = Pipeline([
    ('encoder', OneHotEncoder(handle_unknown='ignore')),
])

print(nominal_pipeline)
print(spaceship.select_dtypes(include = "number").head())

Pipeline(steps=[('encoder', OneHotEncoder(handle_unknown='ignore'))])
   CryoSleep   Age  VIP  RoomService  FoodCourt  ShoppingMall     Spa  VRDeck
0          0  39.0    0          0.0        0.0           0.0     0.0     0.0
1          0  24.0    0        109.0        9.0          25.0   549.0    44.0
2          0  58.0    1         43.0     3576.0           0.0  6715.0    49.0
3          0  33.0    0          0.0     1283.0         371.0  3329.0   193.0
4          0  16.0    0        303.0       70.0         151.0   565.0     2.0


In [68]:
numeric_features = ['CryoSleep','Age','VIP','RoomService','FoodCourt','ShoppingMall','Spa','VRDeck']
nominal_features = ['HomePlanet','Cabin','Destination']


# Branch 1: numerical variables — impute missing values, then scale.
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', MinMaxScaler()),
])

print(numeric_pipeline)

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', MinMaxScaler())])


**Model Selection**

In this exercise we will be using **KNN** as our predictive model.

In [70]:
# Combine the two previously defined preprocessing branches.
preprocessor = ColumnTransformer([
    ('numeric', numeric_pipeline, numeric_features),
    ('nominal', nominal_pipeline, nominal_features)
])

# Complete workflow: preprocessing transformer followed by a predictive model.
booking_pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('model', KNeighborsClassifier(n_neighbors=3)),
])

booking_pipeline.fit(X_train, y_train)
print('Training accuracy:', round(booking_pipeline.score(X_train, y_train), 3))
print('Test accuracy:', round(booking_pipeline.score(X_test, y_test), 3))


Training accuracy: 0.846
Test accuracy: 0.742


- Evaluate your model's performance. Comment it

In [ ]:
This model can intrepet the data with 0.742 accuracy and train data with 0.846 accuracy which are high in values means a good model .